# Sprint 4: Descarga de tarballs DIRECTO A DRIVE

## Por qué este notebook es difer

Los anteriores descargaban a `/content/` — que se borra cuando se apaga
la computadora o se cierra Colab. Este notebook descarga **directamente
a Drive**, que persiste siempre.

## Cómo funciona

1. Cada tarball se descarga a `Drive/Tesis_CXR/data/raw/nih/`.
2. Si tu computadora se apaga a mitad, el archivo parcial queda en Drive.
3. Al reiniciar, `wget -c` retoma la descarga exactamente donde quedó.
4. Una vez descargado y verificado, se procesa y agrega al HDF5.
5. El tarball se borra de Drive para liberar espacio.

## Importante

- **UN TARBALL A LA VEZ.** Corre la celda de descarga, espera que termine,
  luego corre la de procesamiento. Repite para cada uno.
- Cada tarball ocupa ~3.7 GB en Drive temporalmente.
- Necesitas ~4 GB libres en Drive en todo momento.

**Runtime: CPU.**


## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

PROJECT_ROOT = '/content/drive/MyDrive/Tesis_CXR'
NIH_DIR      = f'{PROJECT_ROOT}/data/raw/nih'
H5_DRIVE     = f'{NIH_DIR}/nih_images.h5'
H5_LOCAL     = '/content/nih_images.h5'

import os, shutil, time, tarfile, io, h5py
import numpy as np
from PIL import Image

!pip install -q h5py tqdm
from tqdm import tqdm

# Estado actual del HDF5
with h5py.File(H5_DRIVE, 'r') as hf:
    n = len(hf['images'])
    existing_names = set(hf['images'].keys())

size_gb = os.path.getsize(H5_DRIVE) / (1024**3)
print(f'HDF5 en Drive: {n:,} imágenes | {size_gb:.2f} GB')

# Tarballs pendientes (los que NO tienen sus imágenes en el HDF5)
# Heurística: si el HDF5 tiene >5000 imgs del rango del tarball, ya fue procesado
def tarball_done(tarball_num: int, existing: set) -> bool:
    """Verifica si las imágenes de este tarball ya están en el HDF5."""
    # Cada tarball del NIH tiene imágenes con nombres 0XXXXXXXX_NNN.png
    # El rango aproximado por tarball:
    ranges = {
        6: (6000000, 7500000), 7: (7500000, 9000000),
        8: (9000000, 10500000), 9: (10500000, 12000000),
        10:(12000000,13500000), 11:(13500000,15000000),
        12:(15000000,16500000),
    }
    lo, hi = ranges.get(tarball_num, (0, 99999999))
    count = sum(1 for name in existing
                if lo <= int(name.split('_')[0]) <= hi)
    return count > 3000  # si hay más de 3000 imgs de ese rango, ya fue procesado

print()
print('Estado de los tarballs:')
for i in range(6, 13):
    tar_path = f'{NIH_DIR}/images_{i:03d}.tar.gz'
    done = tarball_done(i, existing_names)
    has_tar = os.path.exists(tar_path)
    size = f'{os.path.getsize(tar_path)/(1024**3):.1f}GB' if has_tar else ''
    status = '✓ procesado' if done else ('📥 descargado' if has_tar else '⬜ pendiente')
    print(f'  images_{i:03d}.tar.gz: {status} {size}')


## 2. Descargar UN tarball a Drive

**Ejecuta esta celda para cada tarball pendiente.**
Cambia el número al inicio de la celda (6, 7, 8... 12).

Si tu computadora se apaga, vuelve a ejecutar esta celda —
`wget -c` retoma desde donde quedó.


In [ ]:
import os

PROJECT_ROOT = '/content/drive/MyDrive/Tesis_CXR'
NIH_DIR      = f'{PROJECT_ROOT}/data/raw/nih'

MIN_SIZE = 3_500_000_000  # ~3.5 GB

print("Verificando tarballs 010-012...\n")

for i in range(10, 13):

    tar_name = f'images_{i:03d}.tar.gz'
    tar_path = f'{NIH_DIR}/{tar_name}'

    print(f'{tar_name}')

    if os.path.exists(tar_path):

        size_bytes = os.path.getsize(tar_path)
        size_gb = size_bytes / (1024**3)

        print(f'  Tamaño: {size_gb:.2f} GB')

        if size_bytes >= MIN_SIZE * 0.95:
            print('  ✓ COMPLETO')
        else:
            print('  ⚠ PARCIAL / INCOMPLETO')

    else:
        print('  ✗ NO EXISTE')

    print()

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# DESCARGAR TARBALLS 010-012
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

BASE_URL = 'https://archive.org/download/ChestXray-NIHCC/ChestXray-NIHCC'
MIN_SIZE = 3_500_000_000  # 3.5 GB mínimo

for TARBALL_NUM in range(10, 13):

    print("=" * 60)

    tar_name  = f'images_{TARBALL_NUM:03d}.tar.gz'
    tar_drive = f'{NIH_DIR}/{tar_name}'
    url       = f'{BASE_URL}/{tar_name}'

    print(f'Tarball: {tar_name}')
    print(f'Destino: {tar_drive}')
    print(f'URL: {url}')
    print()

    # verificar si ya existe
    if os.path.exists(tar_drive):

        current_size = os.path.getsize(tar_drive)
        current_gb = current_size / (1024**3)

        if current_size >= MIN_SIZE * 0.95:
            print(f'✓ Ya descargado y completo ({current_gb:.2f} GB)')
            continue

        else:
            print(f'⚠ Descarga parcial detectada: {current_gb:.2f} GB')
            print('Retomando descarga con wget -c...')
    else:
        print(f'Iniciando descarga de {tar_name}...')
        print()

    t0 = time.time()

    os.system(
        f'wget -c --progress=bar:force:noscroll '
        f'-O "{tar_drive}" "{url}"'
    )

    elapsed = time.time() - t0

    if os.path.exists(tar_drive):
        final_size = os.path.getsize(tar_drive) / (1024**3)
        print(f'\n✓ Descargado: {final_size:.2f} GB en {elapsed/60:.0f} min')
    else:
        print('\n✗ Error en descarga')

    print()

In [ ]:
import os

path = "/content/drive/MyDrive/Tesis_CXR/data/raw/nih/images_012.tar.gz"

if os.path.exists(path):
    os.remove(path)
    print("images_012.tar.gz eliminado.")
else:
    print("No existe.")

In [ ]:
import os, time

BASE_URL = 'https://archive.org/download/ChestXray-NIHCC/ChestXray-NIHCC'

tar_name  = 'images_012.tar.gz'
tar_drive = '/content/drive/MyDrive/Tesis_CXR/data/raw/nih/images_012.tar.gz'
url       = f'{BASE_URL}/{tar_name}'

print("Descargando images_012.tar.gz limpio...")
print()

t0 = time.time()

os.system(
    f'wget --progress=bar:force:noscroll '
    f'-O "{tar_drive}" "{url}"'
)

elapsed = time.time() - t0

if os.path.exists(tar_drive):
    final_size = os.path.getsize(tar_drive)/(1024**3)
    print(f'\nFinal: {final_size:.2f} GB')
    print(f'Tiempo: {elapsed/60:.1f} min')
else:
    print("Error descarga.")

## 3. Verificar integridad del tarball

In [ ]:
tar_name  = f'images_{TARBALL_NUM:03d}.tar.gz'
tar_drive = f'{NIH_DIR}/{tar_name}'

print(f'Verificando integridad de {tar_name}...')
try:
    with tarfile.open(tar_drive, 'r:gz') as tar:
        members = [m for m in tar.getmembers()
                   if m.isfile() and m.name.endswith('.png')]
    print(f'✓ Íntegro: {len(members):,} archivos PNG')

    # Cuántos ya están en el HDF5
    with h5py.File(H5_DRIVE, 'r') as hf:
        existing = set(hf['images'].keys())
    new_count = sum(1 for m in members
                    if os.path.basename(m.name) not in existing)
    print(f'  Ya en HDF5: {len(members)-new_count:,}')
    print(f'  Nuevos a agregar: {new_count:,}')

    if new_count == 0:
        print('\n✓ Todas las imágenes ya están en el HDF5.')
        print('  Puedes pasar al siguiente tarball.')
    else:
        print('\n→ Listo para procesar. Ejecuta la celda 4.')
except Exception as e:
    print(f'✗ Error: {e}')
    print('  El tarball puede estar corrupto. Bórralo y vuelve a descargar.')


In [ ]:
# restaurar h5 sano
import os, shutil

H5_LOCAL = "/content/nih_images.h5"
H5_DRIVE = "/content/drive/MyDrive/Tesis_CXR/data/raw/nih/nih_images.h5"

if os.path.exists(H5_DRIVE):
    os.remove(H5_DRIVE)

shutil.copy2(H5_LOCAL, H5_DRIVE)

print("HDF5 restaurado.")

## 4. Procesar: agregar imágenes al HDF5

Lee las imágenes del tarball en Drive y las agrega al HDF5.
Al terminar, actualiza el HDF5 en Drive y borra el tarball.

**Tiempo estimado: ~15-25 min.**


In [ ]:
tar_name  = f'images_{TARBALL_NUM:03d}.tar.gz'
tar_drive = f'{NIH_DIR}/{tar_name}'

# 1. Copiar HDF5 de Drive a local para escritura rápida
print('Copiando HDF5 a disco local para escritura...')
if os.path.exists(H5_LOCAL):
    # Usar el local si ya está actualizado
    with h5py.File(H5_LOCAL, 'r') as hf_l, h5py.File(H5_DRIVE, 'r') as hf_d:
        n_local = len(hf_l['images'])
        n_drive = len(hf_d['images'])
    if n_local >= n_drive:
        print(f'✓ HDF5 local OK ({n_local:,} imgs). Usando el local.')
    else:
        shutil.copy2(H5_DRIVE, H5_LOCAL)
        print(f'✓ Copiado desde Drive ({n_drive:,} imgs).')
else:
    t0 = time.time()
    shutil.copy2(H5_DRIVE, H5_LOCAL)
    elapsed = time.time() - t0
    with h5py.File(H5_LOCAL, 'r') as hf:
        print(f'✓ Copiado en {elapsed/60:.1f} min ({len(hf["images"]):,} imgs).')

with h5py.File(H5_LOCAL, 'r') as hf:
    existing = set(hf['images'].keys())
print(f'Imágenes actuales en HDF5: {len(existing):,}')

# 2. Agregar imágenes del tarball al HDF5 local
print(f'\nAgregando imágenes de {tar_name}...')
saved, failed = 0, 0
t0 = time.time()

with tarfile.open(tar_drive, 'r:gz') as tar,      h5py.File(H5_LOCAL, 'a') as hf:
    grp = hf.require_group('images')
    to_add = [m for m in tar.getmembers()
              if m.isfile() and m.name.endswith('.png')
              and os.path.basename(m.name) not in existing]
    print(f'Imágenes a agregar: {len(to_add):,}')

    for m in tqdm(to_add, desc=tar_name, unit='img'):
        img_name = os.path.basename(m.name)
        try:
            f = tar.extractfile(m)
            if f is None: continue
            with Image.open(io.BytesIO(f.read())) as pil_img:
                arr = np.array(
                    pil_img.convert('L').resize((256, 256), Image.LANCZOS),
                    dtype=np.uint8
                )
            grp.create_dataset(img_name, data=arr,
                               compression='gzip', compression_opts=4)
            existing.add(img_name)
            saved += 1
        except Exception:
            failed += 1

elapsed = time.time() - t0
print(f'\n✓ Agregadas: {saved:,} | Fallidas: {failed} | Tiempo: {elapsed/60:.0f} min')

# 3. Guardar HDF5 actualizado en Drive
print('\nGuardando HDF5 actualizado en Drive...')
t0 = time.time()
shutil.copy2(H5_LOCAL, H5_DRIVE)
elapsed = time.time() - t0
with h5py.File(H5_DRIVE, 'r') as hf:
    n_total = len(hf['images'])
size_gb = os.path.getsize(H5_DRIVE) / (1024**3)
print(f'✓ HDF5 guardado en Drive: {n_total:,} imgs | {size_gb:.2f} GB | {elapsed/60:.0f} min')

# 4. Borrar tarball de Drive para liberar espacio
os.remove(tar_drive)
print(f'✓ Tarball {tar_name} borrado de Drive.')

print()
print('='*55)
print(f'TARBALL {TARBALL_NUM:03d} PROCESADO ✓')
print(f'  Total en HDF5: {n_total:,} imágenes')
remaining = list(range(TARBALL_NUM + 1, 13))
if remaining:
    print(f'  Pendientes: {[f"{i:03d}" for i in remaining]}')
    print(f'  → Cambia TARBALL_NUM a {remaining[0]} y ejecuta celda 2.')
else:
    print('  ✓ TODOS LOS TARBALLS PROCESADOS')
    print('  → Ejecuta celda 5 para actualizar el CSV.')
print('='*55)


In [ ]:
import h5py

H5_LOCAL = "/content/nih_images.h5"

with h5py.File(H5_LOCAL, "r") as hf:
    total = len(hf["images"])

print("Total en H5 local:", total)

In [ ]:
import h5py

H5_LOCAL = "/content/nih_images.h5"

with h5py.File(H5_LOCAL, "r") as hf:

    existing = set(hf["images"].keys())

print(f"Total imágenes: {len(existing):,}\n")

ranges = {
    1: (0, 1500000),
    2: (1500000, 3000000),
    3: (3000000, 4500000),
    4: (4500000, 6000000),
    5: (6000000, 7500000),
    6: (7500000, 9000000),
    7: (9000000, 10500000),
    8: (10500000, 12000000),
    9: (12000000, 13500000),
    10: (13500000, 15000000),
    11: (15000000, 16500000),
    12: (16500000, 99999999),
}

for tar_num, (lo, hi) in ranges.items():

    count = 0

    for name in existing:
        try:
            num = int(name.split("_")[0])

            if lo <= num < hi:
                count += 1

        except:
            pass

    status = (
        "✓ PROCESADO"
        if count > 8000
        else ("⚠ PARCIAL" if count > 0 else "✗ VACÍO")
    )

    print(f"Tarball {tar_num:03d}: {count:5d} -> {status}")

## 5. Actualizar CSV (solo al terminar todos los tarballs)

In [ ]:
import pandas as pd

# Verificar estado
with h5py.File(H5_DRIVE, 'r') as hf:
    in_h5 = set(hf['images'].keys())
    n = len(in_h5)
print(f'HDF5 final: {n:,} imágenes')

df_full = pd.read_csv(f'{NIH_DIR}/Data_Entry_2017.csv')
df_subset = df_full[df_full['Image Index'].isin(in_h5)].copy()
df_subset.to_csv(f'{NIH_DIR}/Data_Entry_subset_local.csv', index=False)
print(f'CSV actualizado: {len(df_subset):,} filas')

# Distribución
CLASS_NAMES = {0:'No Finding', 1:'Cardiomegaly', 2:'Effusion', 3:'Infiltration'}

def assign_label(finding):
    for kw, lbl in [('Infiltration',3),('Effusion',2),('Cardiomegaly',1)]:
        if kw in finding: return lbl
    return 0 if finding == 'No Finding' else -1

df_subset['label'] = df_subset['Finding Labels'].apply(assign_label)
df_valid = df_subset[df_subset['label'] >= 0]
pat_labels = df_valid.groupby('Patient ID')['label'].max()

print()
print('DISTRIBUCIÓN FINAL:')
print('='*55)
for lbl, name in CLASS_NAMES.items():
    n_p = (pat_labels == lbl).sum()
    print(f'{name:<20}: {n_p:>6,} pacientes únicos')

ceiling = min((pat_labels == l).sum() for l in [1,2,3])
print(f'\nTecho: {ceiling:,} pacientes | Train estimado: ~{ceiling*12:,} imágenes')


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# PROCESAR TARBALLS 010 Y 011
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

for TARBALL_NUM in [10, 11]:

    print()
    print("=" * 60)
    print(f"PROCESANDO TARBALL {TARBALL_NUM:03d}")
    print("=" * 60)

    tar_name  = f'images_{TARBALL_NUM:03d}.tar.gz'
    tar_drive = f'{NIH_DIR}/{tar_name}'

    # 1. Verificar HDF5 local
    print('\nVerificando HDF5 local...')

    if not os.path.exists(H5_LOCAL):
        raise Exception("H5_LOCAL no existe.")

    with h5py.File(H5_LOCAL, 'r') as hf:
        existing = set(hf['images'].keys())

    print(f'Imágenes actuales en HDF5: {len(existing):,}')

    # 2. Procesar tarball
    print(f'\nAgregando imágenes de {tar_name}...')

    saved, failed = 0, 0
    t0 = time.time()

    with tarfile.open(tar_drive, 'r:gz') as tar, \
         h5py.File(H5_LOCAL, 'a') as hf:

        grp = hf.require_group('images')

        to_add = [
            m for m in tar.getmembers()
            if m.isfile()
            and m.name.endswith('.png')
            and os.path.basename(m.name) not in existing
        ]

        print(f'Imágenes a agregar: {len(to_add):,}')

        for m in tqdm(to_add, desc=tar_name, unit='img'):

            img_name = os.path.basename(m.name)

            try:
                f = tar.extractfile(m)

                if f is None:
                    continue

                with Image.open(io.BytesIO(f.read())) as pil_img:

                    arr = np.array(
                        pil_img.convert('L').resize(
                            (256, 256),
                            Image.LANCZOS
                        ),
                        dtype=np.uint8
                    )

                grp.create_dataset(
                    img_name,
                    data=arr,
                    compression='gzip',
                    compression_opts=4
                )

                existing.add(img_name)
                saved += 1

            except Exception:
                failed += 1

    elapsed = time.time() - t0

    print(f'\n✓ Agregadas: {saved:,}')
    print(f'✓ Fallidas: {failed}')
    print(f'✓ Tiempo: {elapsed/60:.1f} min')

    # 3. Verificar total
    with h5py.File(H5_LOCAL, 'r') as hf:
        n_total = len(hf['images'])

    print(f'\n✓ Total actual HDF5 local: {n_total:,}')

    # 4. Borrar tarball procesado
    os.remove(tar_drive)

    print(f'✓ Tarball {tar_name} borrado.')

print()
print("=" * 60)
print("✓ TARBALLS 010 Y 011 PROCESADOS")
print("✓ AÚN NO SE COPIA A DRIVE")
print("=" * 60)

In [ ]:
import os, h5py

H5_LOCAL = "/content/nih_images.h5"
H5_DRIVE = "/content/drive/MyDrive/Tesis_CXR/data/raw/nih/nih_images.h5"

print("Existe local:", os.path.exists(H5_LOCAL))
print("Existe Drive:", os.path.exists(H5_DRIVE))


if os.path.exists(H5_LOCAL):
    print("Tamaño local GB:", os.path.getsize(H5_LOCAL)/(1024**3))

if os.path.exists(H5_DRIVE):
    print("Tamaño Drive GB:", os.path.getsize(H5_DRIVE)/(1024**3))


In [ ]:
import h5py

H5_LOCAL = "/content/nih_images.h5"

with h5py.File(H5_LOCAL, "r") as hf:
    total = len(hf["images"])
    print("LOCAL OK")
    print("Imágenes:", total)



In [ ]:
import os, shutil

H5_LOCAL = "/content/nih_images.h5"
H5_DRIVE = "/content/drive/MyDrive/Tesis_CXR/data/raw/nih/nih_images.h5"

# borrar corrupto
if os.path.exists(H5_DRIVE):
    os.remove(H5_DRIVE)

# copiar sano
shutil.copy2(H5_LOCAL, H5_DRIVE)

print("HDF5 restaurado correctamente.")

In [ ]:
import pandas as pd

df_full = pd.read_csv(f'{NIH_DIR}/Data_Entry_2017.csv')
with h5py.File(H5_LOCAL, 'r') as hf:
    in_h5 = set(hf['images'].keys())

df_subset = df_full[df_full['Image Index'].isin(in_h5)].copy()
df_subset.to_csv(f'{NIH_DIR}/Data_Entry_subset_local.csv', index=False)
print(f'CSV actualizado: {len(df_subset):,} filas')
print()

# Distribución de las 4 clases
print('DISTRIBUCIÓN FINAL (12 tarballs):')
print('='*55)
CLASS_NAMES = {0:'No Finding', 1:'Cardiomegaly', 2:'Effusion', 3:'Infiltration'}

def assign_label(finding):
    for kw, lbl in [('Infiltration',3),('Effusion',2),('Cardiomegaly',1)]:
        if kw in finding: return lbl
    return 0 if finding == 'No Finding' else -1

df_subset['label'] = df_subset['Finding Labels'].apply(assign_label)
df_valid = df_subset[df_subset['label'] >= 0]
pat_labels = df_valid.groupby('Patient ID')['label'].max()

for lbl, name in CLASS_NAMES.items():
    n_imgs = (df_subset['label'] == lbl).sum()
    n_pats = (pat_labels == lbl).sum()
    print(f'{name:<20}: {n_imgs:>8,} imágenes | {n_pats:>6,} pacientes únicos')

ceiling = min((pat_labels == l).sum() for l in [1,2,3])
nf_pats = min((pat_labels == 0).sum(), ceiling * 2)
total_pats = ceiling * 3 + nf_pats

print()
print(f'Techo (clase más pequeña): {ceiling:,} pacientes')
print(f'Dataset balanceado estimado:')
print(f'  {ceiling:,} × 3 positivas + {nf_pats:,} NF = {total_pats:,} pacientes')
print(f'  Train (~todas imgs/paciente × ~4): ~{total_pats*4:,} imágenes')
print()

# Comparativa con 5 tarballs
print('Comparativa 5 vs 12 tarballs:')
print(f'  Techo (Cardiomegaly): 260 → {ceiling:,} pacientes (+{ceiling/260:.0f}x)')
print(f'  Train estimado: 3,900 → ~{total_pats*4:,} imágenes')

In [ ]:
import h5py, os

H5_DRIVE = "/content/drive/MyDrive/Tesis_CXR/data/raw/nih/nih_images.h5"

print("Tamaño Drive GB:", os.path.getsize(H5_DRIVE)/(1024**3))

with h5py.File(H5_DRIVE, "r") as hf:
    print("DRIVE OK")
    print("Imágenes en Drive:", len(hf["images"]))



In [ ]:
import subprocess, time, h5py, os

H5_DRIVE = '/content/drive/MyDrive/Tesis_CXR/data/raw/nih/nih_images.h5'

# Forzar flush de Drive
print('Forzando sync con Drive...')
subprocess.run(['sync'], check=True)

# Esperar un momento
time.sleep(10)

# Verificar
size_gb  = os.path.getsize(H5_DRIVE) / (1024**3)
with h5py.File(H5_DRIVE, 'r') as hf:
    n = len(hf['images'])

print(f'HDF5 en Drive:')
print(f'  Tamaño: {size_gb:.2f} GB')
print(f'  Imágenes: {n:,}')

if n >= 100000:
    print('✓ Sync completo. Puedes cerrar la sesión con seguridad.')
else:
    print('⚠ Aún sincronizando. Espera antes de cerrar la sesión.')